In [1]:
#Prepate the generalisability datasets. For each family group produce cross-condition train/test h5ad files.
#HVGs are selected on the training condition only and applied to the test condition. 
#Cross-genotype and cross-sex
#Install requirements
%pip install -r "../Requirements.txt"

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\xanth\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [12]:
#Import libraries
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import scanpy as sc
import sys
import os

In [3]:
#Add the src to the path
sys.path.append(os.path.abspath(os.path.join('..')))

In [ ]:
#Define the input and ouput file paths
group_dir = Path("../data/data_sets")
gen_dir = Path("../data/generalisability")
gen_dir.mkdir(parents=True, exist_ok=True)

fig_dir = Path("../figures/generalisability")
fig_dir.mkdir(parents=True, exist_ok=True)

In [5]:
group_names = ["Clock", "Glia", "Kenyon_cells", "Monoaminergic", "Optic_lobe", "Peptidergic"]

n_hvgs = 2000 #same number of HVGs with the main pipeline
min_cells = 50 #skip the experiment if either condition has fewer cells

genotypes = ["w1118", "DGRP-551"]
sexes = ["male", "female"]

In [15]:
#Age map order
age_map_order = ["eclosion", "1_very_young", "2_very_young", "1_young", "2_young", "mid_young", "mid_aged", "aged"]
min_per_age = 10

In [10]:
def make_experiment(adata, group, dimension, train_val, test_val):
    train=adata[adata.obs[dimension] == train_val].copy()
    test=adata[adata.obs[dimension] == test_val].copy()

    if train.n_obs < min_cells or test.n_obs < min_cells:
        print(f"Skip {dimension}")
        return
    
    #HVGs on the training set only
    sc.pp.highly_variable_genes(train, n_top_genes=n_hvgs)
    hvg = train.var_names[train.var["highly_variable"]].tolist()
    
    #Apply the same HVGs list to the train and test set
    train_hvg = train[:,hvg].copy()
    test_hvg = test[:,hvg].copy()

    tag = f"{dimension.lower()}_train_{train_val}_test_{test_val}".replace("/","_")
    train_hvg.write_h5ad(gen_dir / f"{group}__{tag}_train.h5ad")
    test_hvg.write_h5ad(gen_dir / f"{group}__{tag}_test.h5ad")

    print(f"{tag}: train and test sets saved")

In [11]:
#Loop

for group in group_names:
    gfile = group_dir / f"{group}.h5ad"
    if not gfile.exists():
        print(f"Warning: {gfile.name} not found")
        continue

    adata=sc.read_h5ad(gfile)
    print(f"Genotype: {adata.obs['Genotype'].value_counts().to_dict()}")
    print(f"Sex: {adata.obs['sex'].value_counts().to_dict()}")

    #cross-genotype
    make_experiment(adata, group, "Genotype", genotypes[0], genotypes[1])
    make_experiment(adata, group, "Genotype", genotypes[1], genotypes[0])

    #cross-sex
    make_experiment(adata, group, "sex", sexes[0], sexes[1])
    make_experiment(adata, group, "sex", sexes[1], sexes[0])

    print(f"\n All generalisability h5ad files were saved for: {group}")

Genotype: {'DGRP-551': 272, 'w1118': 244}
Sex: {'female': 278, 'male': 238}
genotype_train_w1118_test_DGRP-551: train and test sets saved
genotype_train_DGRP-551_test_w1118: train and test sets saved
sex_train_male_test_female: train and test sets saved
sex_train_female_test_male: train and test sets saved

 All generalisability h5ad files were saved for: Clock
Genotype: {'DGRP-551': 2386, 'w1118': 1353}
Sex: {'female': 2014, 'male': 1725}
genotype_train_w1118_test_DGRP-551: train and test sets saved
genotype_train_DGRP-551_test_w1118: train and test sets saved
sex_train_male_test_female: train and test sets saved
sex_train_female_test_male: train and test sets saved

 All generalisability h5ad files were saved for: Glia
Genotype: {'w1118': 1530, 'DGRP-551': 1318}
Sex: {'female': 1501, 'male': 1347}
genotype_train_w1118_test_DGRP-551: train and test sets saved
genotype_train_DGRP-551_test_w1118: train and test sets saved
sex_train_male_test_female: train and test sets saved
sex_train_f

In [16]:
#check the experiments for coverage
EXPERIMENTS = [
    ("Genotype", "w1118",    "DGRP-551"),
    ("Genotype", "DGRP-551", "w1118"),
    ("sex",      "male",     "female"),
    ("sex",      "female",   "male"),
]

In [17]:
flag_rows = []
 
for group in group_names:
    gfile = group_dir / f"{group}.h5ad"
    if not gfile.exists():
        print(f"WARNING: {gfile.name} not found, skipping")
        continue
 
    adata = sc.read_h5ad(gfile)
    print("\n" + "="*70)
    print(f"GROUP: {group}  ({adata.n_obs:,} cells)")
    print("="*70)
 
    #one figure per group: 2 rows (genotype, sex) x 2 cols (age, cell type)
    fig, axes = plt.subplots(2, 2, figsize=(15, 9))
 
    for row, (dim, train_val, test_val) in enumerate(
        [("Genotype","w1118","DGRP-551"), ("sex","male","female")]
    ):
        train = adata[adata.obs[dim] == train_val]
        test  = adata[adata.obs[dim] == test_val]
 
        # ---- AGE coverage ----
        tr_age = train.obs["age_map"].value_counts().reindex(age_map_order, fill_value=0)
        te_age = test.obs["age_map"].value_counts().reindex(age_map_order, fill_value=0)
 
        x = np.arange(len(age_map_order))
        axes[row,0].bar(x-0.2, tr_age.values, width=0.4,
                        label=f"train ({train_val})", color="#2E75B6")
        axes[row,0].bar(x+0.2, te_age.values, width=0.4,
                        label=f"test ({test_val})", color="#E8733A")
        axes[row,0].axhline(min_per_age, color="red", ls="--", lw=0.8)
        axes[row,0].set_xticks(x)
        axes[row,0].set_xticklabels([a.replace("_","\n") for a in age_map_order], fontsize=7)
        axes[row,0].set_title(f"{dim}: age coverage")
        axes[row,0].set_ylabel("cells")
        axes[row,0].legend(fontsize=8)
 
        #cell-type composition
        tr_ct = train.obs["annotation"].value_counts(normalize=True)
        te_ct = test.obs["annotation"].value_counts(normalize=True)
        all_ct = sorted(set(tr_ct.index) | set(te_ct.index))
        tr_ct = tr_ct.reindex(all_ct, fill_value=0)
        te_ct = te_ct.reindex(all_ct, fill_value=0)
 
        xc = np.arange(len(all_ct))
        axes[row,1].bar(xc-0.2, tr_ct.values, width=0.4,
                        label=f"train ({train_val})", color="#2E75B6")
        axes[row,1].bar(xc+0.2, te_ct.values, width=0.4,
                        label=f"test ({test_val})", color="#E8733A")
        axes[row,1].set_xticks(xc)
        axes[row,1].set_xticklabels(all_ct, rotation=90, fontsize=6)
        axes[row,1].set_title(f"{dim}: cell-type composition (proportion)")
        axes[row,1].set_ylabel("proportion")
        axes[row,1].legend(fontsize=8)
 
        #summary 
        print(f"\n--- {dim}: train={train_val} ({train.n_obs}) | test={test_val} ({test.n_obs}) ---")
        cov = pd.DataFrame({"train": tr_age, "test": te_age})
        print(cov.to_string())
 
        # flag: ages where TRAIN is too sparse (model cannot learn that age)
        weak_train_ages = tr_age[tr_age < min_per_age].index.tolist()
        # flag: cell-type composition divergence (max abs proportion difference)
        ct_divergence = float((tr_ct - te_ct).abs().max())
 
        if weak_train_ages:
            print(f"  FLAG: training set sparse at ages: {weak_train_ages}")
        if ct_divergence > 0.25:
            print(f"  FLAG: cell-type composition differs by up to "
                  f"{ct_divergence:.0%} between conditions (possible confounder)")
 
        flag_rows.append({
            "group": group,
            "dimension": dim,
            "train": train_val, "test": test_val,
            "n_train": train.n_obs, "n_test": test.n_obs,
            "weak_train_ages": ",".join(weak_train_ages) if weak_train_ages else "none",
            "celltype_max_divergence": round(ct_divergence, 2),
            "clean": (len(weak_train_ages) == 0 and ct_divergence <= 0.25),
        })
 
    plt.suptitle(f"{group} — generalisability validity checks", fontsize=14)
    plt.tight_layout()
    plt.savefig(fig_dir / f"{group}_validity_check.png", dpi=200)
    plt.close()
    print(f"\n  Saved figure: {group}_validity_check.png")
 
#master flag table
flags = pd.DataFrame(flag_rows)
flags.to_csv(gen_dir / "validity_flags.csv", index=False)
print("VALIDITY FLAG SUMMARY (clean = safe to interpret directly)")
print(flags.to_string(index=False))
print(f"\nClean experiments: {flags['clean'].sum()} / {len(flags)}")
print("Experiments flagged as confounded should be reported with caveats, or restricted to shared age ranges before running.")



GROUP: Clock  (516 cells)

--- Genotype: train=w1118 (244) | test=DGRP-551 (272) ---
              train  test
age_map                  
eclosion         59    41
1_very_young     22    59
2_very_young     39    27
1_young          68    23
2_young          35    10
mid_young        15    31
mid_aged          6    55
aged              0    26
  FLAG: training set sparse at ages: ['mid_aged', 'aged']

--- sex: train=male (238) | test=female (278) ---
              train  test
age_map                  
eclosion         54    46
1_very_young     45    36
2_very_young     25    41
1_young          41    50
2_young          23    22
mid_young        18    28
mid_aged         22    39
aged             10    16

  Saved figure: Clock_validity_check.png

GROUP: Glia  (3,739 cells)

--- Genotype: train=w1118 (1353) | test=DGRP-551 (2386) ---
              train  test
age_map                  
eclosion        342   370
1_very_young    136   238
2_very_young    155    99
1_young         339   10